# Recency-Aware Retrieval for RAG

## 1. Project Overview

**Task:** Build a retrieval system that **prefers recent documents**, **labels stale information risk**, and explains how recency ranking and metadata filters work.

**Why this matters:** Standard vector search ranks by semantic similarity alone — a 5-year-old policy guide and today's update score the same if the words match. In domains like compliance, finance, medicine, and engineering, **stale information presented as current is actively dangerous**.

This notebook builds:
1. A **time-decay scoring** function that blends semantic similarity with document freshness
2. **Metadata filters** that restrict retrieval to specific date ranges
3. A **staleness labeller** that flags when retrieved evidence may be outdated
4. An evaluation comparing recency-naive vs recency-aware retrieval

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — LLM (local, no API keys)
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — embedding model
- `ChromaDB` — vector store with metadata filtering

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **why semantic similarity alone is insufficient** when documents have timestamps |
| 2 | Implement **time-decay scoring** that blends relevance and freshness |
| 3 | Use **metadata filters** on ChromaDB to restrict retrieval by date |
| 4 | Build a **staleness labeller** that warns when evidence is outdated |
| 5 | Compare **recency-naive vs recency-aware** retrieval quality |
| 6 | Understand the **trade-offs** of different decay functions (linear, exponential, step) |

## 3. The Recency Problem in RAG

### Why Similarity-Only Retrieval Fails

```
QUESTION: "What is our remote work policy?"

SIMILARITY-ONLY RETRIEVAL:                       RECENCY-AWARE RETRIEVAL:
┌─────────────────────────────────┐              ┌─────────────────────────────────┐
│ 1. Remote Work Policy v1 (2020) │ sim=0.94     │ 1. Remote Work Policy v3 (2024) │ combined=0.90
│ 2. Remote Work Policy v3 (2024) │ sim=0.91     │ 2. Remote Work Addendum (2024)  │ combined=0.85
│ 3. Remote Work FAQ (2021)       │ sim=0.89     │ 3. Remote Work Policy v1 (2020) │ combined=0.52
│                                 │              │    ⚠ STALE — 4 years old         │
└─────────────────────────────────┘              └─────────────────────────────────┘
   ↑ Returns outdated v1 as top hit!                ↑ Returns current v3 as top hit
```

### Recency Ranking Approaches

| Approach | How It Works | Pros | Cons |
|----------|-------------|------|------|
| **Hard filter** | Exclude docs older than N days | Simple, deterministic | Loses valid old knowledge |
| **Time-decay scoring** | `score = sim × decay(age)` | Smooth preference for recent | Needs tuning of decay rate |
| **Recency boost** | `score = sim + boost × recency` | Additive, easy to control | Can override high-relevance old docs |
| **Bucketed windows** | "Prefer last 6 months, allow up to 2 years" | Practical for policies | Arbitrary bucket boundaries |

### Metadata Filters

ChromaDB (and most vector stores) support **metadata filtering** — restricting search to documents matching specific conditions *before* similarity ranking:

```python
# Only search documents from 2024 onward
collection.query(
    query_embeddings=[...],
    where={"year": {"$gte": 2024}},
    n_results=5,
)

# Only search a specific category
collection.query(
    query_embeddings=[...],
    where={"category": "policy"},
    n_results=5,
)
```

Filters are applied **before** the vector search, so they're efficient and precise.

## 4. Pipeline Architecture

```
  Question
     │
     ▼
  ┌──────────────────────────┐
  │  Embed query             │
  └──────────┬───────────────┘
             │
     ┌───────┴────────┐
     ▼                ▼
  ┌──────────┐   ┌──────────────┐
  │ Metadata │   │ similarity   │
  │ filter   │   │ search       │
  │ (date)   │   │ (all docs)   │
  └────┬─────┘   └──────┬───────┘
       │                │
       ▼                ▼
  ┌──────────────────────────┐
  │  Time-Decay Reranking    │
  │  score = sim × decay(age)│
  └──────────┬───────────────┘
             │
             ▼
  ┌──────────────────────────┐
  │  Staleness Labeller      │
  │  Flag stale documents    │
  │  Warn if newest < 6 mo  │
  └──────────┬───────────────┘
             │
             ▼
  ┌──────────────────────────┐
  │  LLM Answer Generation   │
  │  + staleness warnings    │
  └──────────────────────────┘
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports & Configuration

In [ ]:
import os
import re
import json
import math
import time
import textwrap
import warnings
from datetime import datetime, timedelta
from collections import Counter

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 5

# Recency settings
STALE_THRESHOLD_DAYS = 365        # Documents older than 1 year get a staleness warning
VERY_STALE_THRESHOLD_DAYS = 730   # Documents older than 2 years are flagged as high risk
DECAY_HALF_LIFE_DAYS = 180        # Exponential decay: score halves every 6 months
REFERENCE_DATE = datetime(2025, 1, 15)  # "Today" for our simulation

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Reference date: {REFERENCE_DATE.strftime('%Y-%m-%d')}")
print(f"Stale threshold: {STALE_THRESHOLD_DAYS} days ({STALE_THRESHOLD_DAYS // 365} year)")
print(f"Decay half-life: {DECAY_HALF_LIFE_DAYS} days ({DECAY_HALF_LIFE_DAYS // 30} months)")

## 7. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Document Corpus with Temporal Metadata

## 8. Build a Time-Stamped Corpus

We simulate a company knowledge base where the **same topic** has documents from different dates. Some are current, some are outdated, and some have been superseded.

In [ ]:
DOCUMENTS = [
    # --- Remote Work Policy (3 versions) ---
    {"id": "D01", "date": "2020-03-15", "category": "policy",
     "title": "Remote Work Policy v1",
     "superseded_by": "D06",
     "text": "Remote work is permitted on a case-by-case basis with manager approval. Employees must be in the office at least 4 days per week. Remote work requests require a written justification and VP-level sign-off. All remote workers must use the company VPN and are expected to be available during core hours (9 AM - 5 PM EST)."},

    {"id": "D06", "date": "2022-01-10", "category": "policy",
     "title": "Remote Work Policy v2",
     "superseded_by": "D14",
     "text": "The company adopts a hybrid model. Employees may work remotely up to 3 days per week without special approval. Manager approval is needed only for fully remote arrangements. Core collaboration hours are Tuesday and Thursday in-office. VPN is required for remote access. Equipment stipend of $500 for home office setup."},

    {"id": "D14", "date": "2024-06-01", "category": "policy",
     "title": "Remote Work Policy v3 — Current",
     "superseded_by": None,
     "text": "Flexible work is the default. Employees choose their work location. One mandatory in-office day per month for team sync. Fully remote roles are available for all positions except hardware lab work. Equipment stipend increased to $1,200. Coworking space reimbursement up to $300/month. Core hours reduced to 10 AM - 3 PM in the employee's local timezone."},

    # --- Security Policy (2 versions + addendum) ---
    {"id": "D02", "date": "2019-07-22", "category": "policy",
     "title": "Information Security Policy v1",
     "superseded_by": "D10",
     "text": "All employees must use passwords of at least 8 characters. Password rotation every 90 days is mandatory. Two-factor authentication is recommended but not required. USB drives are permitted for file transfer. Email attachments up to 25 MB are allowed. Security training is annual."},

    {"id": "D10", "date": "2023-09-15", "category": "policy",
     "title": "Information Security Policy v2 — Current",
     "superseded_by": None,
     "text": "Minimum 16-character passphrases required. Password rotation eliminated per NIST 800-63B guidelines. MFA is mandatory for all accounts — hardware keys preferred, TOTP acceptable. USB drives are banned; use approved cloud storage only. Email DLP scanning is active. Phishing simulation tests run monthly. Security training is quarterly with role-based modules."},

    {"id": "D15", "date": "2024-11-20", "category": "policy",
     "title": "Security Addendum — AI Tool Usage",
     "superseded_by": None,
     "text": "Employees may use approved AI tools (listed in the internal tool registry). Confidential data must not be pasted into external AI services. Self-hosted LLMs are approved for all data classifications. Code generated by AI must pass the same review standards as human-written code. AI-generated content in customer-facing materials must be reviewed by Legal."},

    # --- Compensation (2 versions) ---
    {"id": "D03", "date": "2021-04-01", "category": "compensation",
     "title": "Compensation Framework 2021",
     "superseded_by": "D12",
     "text": "Base salary ranges are tied to city-tier adjustments. Tier 1 (SF, NYC): 100%. Tier 2 (Austin, Denver): 85%. Tier 3 (all others): 75%. Annual raises are 3-5% for meets expectations, 6-10% for exceeds. Stock options vest over 4 years with a 1-year cliff. Bonus pool is 10% of base salary at target."},

    {"id": "D12", "date": "2024-03-01", "category": "compensation",
     "title": "Compensation Framework 2024 — Current",
     "superseded_by": None,
     "text": "Location-based pay differentials eliminated. All roles pay the same regardless of geography. RSU grants replace stock options for new employees. Annual raises are 4-7% meets, 8-15% exceeds. Bonus pool increased to 15% at target with company performance multiplier (0.5x - 2.0x). Equity refresh grants every 2 years. Salary transparency: all bands published internally."},

    # --- Product docs (evolving over time) ---
    {"id": "D04", "date": "2020-11-01", "category": "product",
     "title": "Platform API v1 Documentation",
     "superseded_by": "D09",
     "text": "The REST API uses API key authentication. Rate limit: 100 requests per minute. Endpoints: /users, /data, /reports. All responses are JSON. Pagination via offset/limit parameters. Webhook support for data export events. SDK available for Python and JavaScript. API versioning via URL path (/v1/)."},

    {"id": "D09", "date": "2023-06-15", "category": "product",
     "title": "Platform API v2 Documentation",
     "superseded_by": "D16",
     "text": "API v2 uses OAuth 2.0 with PKCE for authentication. Rate limit increased to 1,000 requests per minute with burst allowance. New endpoints: /analytics, /ml-models, /integrations. GraphQL support added alongside REST. Cursor-based pagination replaces offset/limit. Real-time streaming via Server-Sent Events. SDKs for Python, JavaScript, Go, and Rust."},

    {"id": "D16", "date": "2024-10-01", "category": "product",
     "title": "Platform API v3 Documentation — Current",
     "superseded_by": None,
     "text": "API v3 introduces gRPC alongside REST and GraphQL. Authentication via mTLS for service-to-service and OAuth 2.1 for user-facing apps. Rate limit: 5,000 requests per minute. New endpoints: /workflows, /agents, /knowledge-base. Streaming responses for LLM-powered endpoints. OpenAPI 3.1 spec auto-generated. SDKs for Python, JavaScript, Go, Rust, and Java. API key auth deprecated — migration deadline March 2025."},

    # --- Compliance ---
    {"id": "D05", "date": "2021-08-10", "category": "compliance",
     "title": "GDPR Compliance Guide v1",
     "superseded_by": "D13",
     "text": "Customer data is stored in EU-West-1 (Ireland). Data retention: 7 years for financial records, 3 years for user activity logs. Data subject access requests (DSARs) must be fulfilled within 30 days. Cookie consent banners are required on all web properties. Data Processing Agreements (DPAs) must be signed with all sub-processors."},

    {"id": "D13", "date": "2024-02-15", "category": "compliance",
     "title": "Data Privacy Compliance Guide — Current",
     "superseded_by": None,
     "text": "Compliance now covers GDPR, CCPA, and Brazil's LGPD. Data residency options: EU (Ireland), US (Virginia), APAC (Singapore). Retention reduced: 5 years financial, 1 year activity logs. Automated DSAR fulfillment via self-service portal (average: 2 days). AI model training on customer data requires explicit opt-in consent. Privacy impact assessments required for all new features processing personal data."},

    # --- Engineering guidelines ---
    {"id": "D07", "date": "2022-05-20", "category": "engineering",
     "title": "Code Review Standards",
     "superseded_by": None,
     "text": "All code changes require at least one approval from a team member. Critical paths (auth, payments, data deletion) require two approvals including a senior engineer. PR descriptions must include: what changed, why, how to test, and rollback plan. CI must pass before merge. No force-pushes to main. Feature flags for all user-facing changes."},

    {"id": "D08", "date": "2022-09-01", "category": "engineering",
     "title": "Incident Response Playbook",
     "superseded_by": "D17",
     "text": "Severity levels: SEV1 (full outage), SEV2 (partial degradation), SEV3 (minor impact). SEV1: page on-call within 5 minutes, assemble war room, communicate to customers within 30 minutes. SEV2: respond within 15 minutes, status page update within 1 hour. Post-incident reviews required within 5 business days for SEV1/2. Blameless culture — focus on system improvements."},

    {"id": "D17", "date": "2024-09-10", "category": "engineering",
     "title": "Incident Response Playbook v2 — Current",
     "superseded_by": None,
     "text": "Severity levels updated: SEV0 added for security breaches affecting customer data. SEV0: page CISO and on-call within 2 minutes, Legal notified immediately, customer notification within 4 hours per regulatory requirements. Automated runbooks for common SEV2/3 incidents. AI-powered anomaly detection triggers pre-emptive alerts. Post-incident reviews now include LLM-generated timeline summaries. Incident metrics published monthly."},

    # --- Recent standalone docs ---
    {"id": "D11", "date": "2024-08-05", "category": "product",
     "title": "Q3 2024 Product Roadmap",
     "superseded_by": None,
     "text": "Q3 priorities: (1) Launch knowledge base API for RAG applications, (2) GA release of workflow automation engine, (3) SOC2 Type II recertification, (4) Migrate remaining API v1 users to v3. Key risk: v1 deprecation may cause churn among long-tail customers. Mitigation: extended support window and migration tooling. Team allocation: 60% new features, 30% tech debt, 10% security."},

    {"id": "D18", "date": "2024-12-01", "category": "product",
     "title": "2025 Strategic Plan",
     "superseded_by": None,
     "text": "2025 focus areas: (1) AI-native features in every product surface, (2) Expand to APAC market with Singapore data center, (3) Launch marketplace for community-built integrations, (4) Achieve FedRAMP Moderate authorization. Revenue target: $50M ARR (up from $35M). Headcount growth: 40% engineering, 25% go-to-market. Key bet: agentic AI workflows as premium tier."},
]

# Parse dates and compute ages
for doc in DOCUMENTS:
    doc["date_obj"] = datetime.strptime(doc["date"], "%Y-%m-%d")
    doc["age_days"] = (REFERENCE_DATE - doc["date_obj"]).days
    doc["age_months"] = round(doc["age_days"] / 30.44)
    doc["is_current"] = doc["superseded_by"] is None

print(f"Corpus: {len(DOCUMENTS)} documents")
print(f"Date range: {min(d['date'] for d in DOCUMENTS)} to {max(d['date'] for d in DOCUMENTS)}")
print(f"Categories: {dict(Counter(d['category'] for d in DOCUMENTS))}")
print(f"Current (not superseded): {sum(1 for d in DOCUMENTS if d['is_current'])}")
print(f"Superseded (outdated): {sum(1 for d in DOCUMENTS if not d['is_current'])}")
print(f"\nAge distribution:")
for d in sorted(DOCUMENTS, key=lambda x: x["date"]):
    status = "CURRENT" if d["is_current"] else f"replaced by {d['superseded_by']}"
    print(f"  {d['date']}  [{d['id']}] {d['title'][:38]:<40} age={d['age_months']:>2}mo  {status}")

## 9. Index Documents in ChromaDB

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="docs", metadata={"hnsw:space": "cosine"},
)

# Index with rich metadata for filtering
texts = [d["text"] for d in DOCUMENTS]
metas = [
    {
        "doc_id": d["id"],
        "title": d["title"],
        "date": d["date"],
        "category": d["category"],
        "age_days": d["age_days"],
        "is_current": d["is_current"],
        # ChromaDB needs numeric for comparisons
        "year": d["date_obj"].year,
        "date_ordinal": d["date_obj"].toordinal(),
    }
    for d in DOCUMENTS
]
ids = [d["id"] for d in DOCUMENTS]
vecs = embeddings.embed_documents(texts)
collection.add(documents=texts, embeddings=vecs, metadatas=metas, ids=ids)

print(f"Indexed {collection.count()} documents with temporal metadata")
print(f"Metadata fields: doc_id, title, date, category, age_days, is_current, year, date_ordinal")

---

# Part B — Retrieval Methods

## 10. Baseline — Similarity-Only Retrieval

In [ ]:
def retrieve_similarity_only(query: str, top_k: int = TOP_K) -> list[dict]:
    """Standard vector search — ranked by semantic similarity only."""
    qv = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[qv], n_results=top_k)
    hits = []
    for i in range(len(results["documents"][0])):
        sim = round(1 - results["distances"][0][i], 4)
        meta = results["metadatas"][0][i]
        hits.append({
            "doc_id": meta["doc_id"],
            "title": meta["title"],
            "date": meta["date"],
            "category": meta["category"],
            "age_days": meta["age_days"],
            "is_current": meta["is_current"],
            "text": results["documents"][0][i],
            "similarity": sim,
            "final_score": sim,  # No recency adjustment
            "method": "similarity_only",
        })
    return hits


# Demo: see how similarity-only ranks old docs high
test_q = "What is our remote work policy?"
hits = retrieve_similarity_only(test_q)
print(f"Q: {test_q}")
print(f"{'Rank':<5} {'Score':<7} {'Date':<12} {'Current?':<10} {'Title'}")
print("-" * 75)
for i, h in enumerate(hits, 1):
    curr = "YES" if h["is_current"] else "NO"
    print(f"{i:<5} {h['final_score']:<7.3f} {h['date']:<12} {curr:<10} {h['title'][:40]}")

## 11. Time-Decay Functions

Three different approaches to penalizing old documents.

In [ ]:
def decay_exponential(age_days: int, half_life: int = DECAY_HALF_LIFE_DAYS) -> float:
    """Exponential decay: score halves every `half_life` days.
    Smooth, natural decay. Documents gradually lose freshness weight.
    
    decay = 0.5 ^ (age / half_life)
    """
    return 0.5 ** (age_days / half_life)


def decay_linear(age_days: int, max_age: int = 1095) -> float:
    """Linear decay: drops to 0 at max_age days (default 3 years).
    Simple but creates a hard cutoff.
    
    decay = max(0, 1 - age/max_age)
    """
    return max(0.0, 1.0 - age_days / max_age)


def decay_step(age_days: int) -> float:
    """Step decay: full score for recent, reduced for older.
    Practical for policy-like use cases with clear freshness tiers.
    
    < 6 months: 1.0
    6-12 months: 0.85
    1-2 years: 0.6
    2-3 years: 0.35
    > 3 years: 0.15
    """
    if age_days < 180:
        return 1.0
    elif age_days < 365:
        return 0.85
    elif age_days < 730:
        return 0.6
    elif age_days < 1095:
        return 0.35
    else:
        return 0.15


# Visualize decay functions across document ages
print("DECAY FUNCTION COMPARISON")
print(f"{'Age':<12} {'Exponential':<14} {'Linear':<14} {'Step':<14}")
print("-" * 54)
for age in [0, 30, 90, 180, 365, 548, 730, 1095, 1460, 1825]:
    e = decay_exponential(age)
    l = decay_linear(age)
    s = decay_step(age)
    label = f"{age}d ({age // 30}mo)"
    print(f"{label:<12} {e:<14.3f} {l:<14.3f} {s:<14.3f}")

print(f"\nFor this notebook we use EXPONENTIAL decay with {DECAY_HALF_LIFE_DAYS}-day half-life.")

## 12. Recency-Aware Retrieval

In [ ]:
def retrieve_recency_aware(
    query: str,
    top_k: int = TOP_K,
    decay_fn=decay_exponential,
    recency_weight: float = 0.35,
) -> list[dict]:
    """Retrieve with combined similarity + recency scoring.
    
    final_score = (1 - recency_weight) * similarity + recency_weight * decay(age)
    
    recency_weight controls how much freshness matters:
      0.0 = pure similarity (ignores dates)
      0.5 = equal blend
      1.0 = pure recency (ignores content)
    """
    # Fetch more candidates than needed, then rerank
    fetch_k = min(len(DOCUMENTS), top_k * 3)
    qv = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[qv], n_results=fetch_k)

    hits = []
    for i in range(len(results["documents"][0])):
        sim = 1 - results["distances"][0][i]
        meta = results["metadatas"][0][i]
        age_days = meta["age_days"]
        decay_score = decay_fn(age_days)

        # Weighted combination
        combined = (1 - recency_weight) * sim + recency_weight * decay_score

        hits.append({
            "doc_id": meta["doc_id"],
            "title": meta["title"],
            "date": meta["date"],
            "category": meta["category"],
            "age_days": age_days,
            "is_current": meta["is_current"],
            "text": results["documents"][0][i],
            "similarity": round(sim, 4),
            "decay": round(decay_score, 4),
            "final_score": round(combined, 4),
            "method": "recency_aware",
        })

    # Rerank by combined score
    hits.sort(key=lambda x: x["final_score"], reverse=True)
    return hits[:top_k]


# Compare side by side
test_q = "What is our remote work policy?"
naive_hits = retrieve_similarity_only(test_q)
aware_hits = retrieve_recency_aware(test_q)

print(f"Q: {test_q}")
print(f"\n  SIMILARITY ONLY:")
for i, h in enumerate(naive_hits[:3], 1):
    curr = "CURRENT" if h["is_current"] else "STALE"
    print(f"    {i}. [{h['doc_id']}] score={h['final_score']:.3f}  {h['date']}  {curr:<8} {h['title'][:35]}")

print(f"\n  RECENCY-AWARE (weight=0.35):")
for i, h in enumerate(aware_hits[:3], 1):
    curr = "CURRENT" if h["is_current"] else "STALE"
    print(f"    {i}. [{h['doc_id']}] score={h['final_score']:.3f} (sim={h['similarity']:.3f} decay={h['decay']:.3f})  {h['date']}  {curr:<8} {h['title'][:35]}")

## 13. Metadata-Filtered Retrieval

In [ ]:
def retrieve_filtered(
    query: str,
    top_k: int = TOP_K,
    category: str | None = None,
    min_year: int | None = None,
    max_age_days: int | None = None,
    current_only: bool = False,
) -> list[dict]:
    """Retrieve with metadata filters applied BEFORE vector search."""
    qv = embeddings.embed_query(query)

    # Build where clause
    conditions = []
    if category:
        conditions.append({"category": {"$eq": category}})
    if min_year:
        conditions.append({"year": {"$gte": min_year}})
    if max_age_days:
        conditions.append({"age_days": {"$lte": max_age_days}})
    if current_only:
        conditions.append({"is_current": {"$eq": True}})

    where = None
    if len(conditions) == 1:
        where = conditions[0]
    elif len(conditions) > 1:
        where = {"$and": conditions}

    try:
        results = collection.query(
            query_embeddings=[qv],
            n_results=top_k,
            where=where,
        )
    except Exception as e:
        print(f"  Filter error: {e}")
        results = collection.query(query_embeddings=[qv], n_results=top_k)

    hits = []
    for i in range(len(results["documents"][0])):
        sim = round(1 - results["distances"][0][i], 4)
        meta = results["metadatas"][0][i]
        hits.append({
            "doc_id": meta["doc_id"],
            "title": meta["title"],
            "date": meta["date"],
            "category": meta["category"],
            "age_days": meta["age_days"],
            "is_current": meta["is_current"],
            "text": results["documents"][0][i],
            "similarity": sim,
            "final_score": sim,
            "method": "filtered",
        })
    return hits


# Demo: metadata filters
print("METADATA FILTER EXAMPLES")
print("=" * 80)

# Filter 1: Only current (not superseded) docs
h1 = retrieve_filtered("remote work policy", current_only=True, top_k=3)
print(f"\n  current_only=True: 'remote work policy'")
for h in h1:
    print(f"    [{h['doc_id']}] {h['date']}  sim={h['similarity']:.3f}  {h['title'][:40]}")

# Filter 2: Only 2024+ docs
h2 = retrieve_filtered("API documentation", min_year=2024, top_k=3)
print(f"\n  min_year=2024: 'API documentation'")
for h in h2:
    print(f"    [{h['doc_id']}] {h['date']}  sim={h['similarity']:.3f}  {h['title'][:40]}")

# Filter 3: Category + recency
h3 = retrieve_filtered("security requirements", category="policy", max_age_days=730, top_k=3)
print(f"\n  category=policy, max_age=730d: 'security requirements'")
for h in h3:
    print(f"    [{h['doc_id']}] {h['date']}  sim={h['similarity']:.3f}  {h['title'][:40]}")

---

# Part C — Staleness Labeller

## 14. Staleness Risk Classification

Every retrieved document gets a staleness label so the LLM (and the user) know how much to trust it.

In [ ]:
def label_staleness(hit: dict) -> dict:
    """Add staleness risk label and warning message to a retrieval hit."""
    age = hit["age_days"]
    is_current = hit["is_current"]

    # Superseded docs are always stale regardless of age
    if not is_current:
        hit["staleness"] = "SUPERSEDED"
        hit["staleness_risk"] = "HIGH"
        hit["staleness_warning"] = (
            f"This document has been replaced by a newer version. "
            f"Information may be outdated or incorrect."
        )
        return hit

    if age <= 180:  # < 6 months
        hit["staleness"] = "FRESH"
        hit["staleness_risk"] = "NONE"
        hit["staleness_warning"] = ""
    elif age <= STALE_THRESHOLD_DAYS:  # 6-12 months
        hit["staleness"] = "AGING"
        hit["staleness_risk"] = "LOW"
        hit["staleness_warning"] = (
            f"Document is {age // 30} months old. Content is likely current but "
            f"verify for any recent changes."
        )
    elif age <= VERY_STALE_THRESHOLD_DAYS:  # 1-2 years
        hit["staleness"] = "STALE"
        hit["staleness_risk"] = "MEDIUM"
        hit["staleness_warning"] = (
            f"Document is {age // 30} months old ({age // 365} year+). "
            f"Details may have changed. Check for updated versions."
        )
    else:  # > 2 years
        hit["staleness"] = "VERY_STALE"
        hit["staleness_risk"] = "HIGH"
        hit["staleness_warning"] = (
            f"Document is {age // 365} years old. HIGH RISK of outdated information. "
            f"Do not rely on specifics without verification."
        )
    return hit


def label_result_set(hits: list[dict]) -> list[dict]:
    """Label all hits and add a summary staleness assessment."""
    for h in hits:
        label_staleness(h)
    return hits


# Demo
demo_hits = retrieve_similarity_only("What is our security policy?", top_k=5)
demo_hits = label_result_set(demo_hits)

print("STALENESS LABELS")
print("=" * 90)
for h in demo_hits:
    icon = {"NONE": "✓", "LOW": "~", "MEDIUM": "!", "HIGH": "✗"}[h["staleness_risk"]]
    print(f"  [{icon}] {h['staleness']:<12} risk={h['staleness_risk']:<6}  {h['date']}  {h['title'][:40]}")
    if h["staleness_warning"]:
        print(f"      {h['staleness_warning']}")

## 15. Full Recency-Aware QA Pipeline

In [ ]:
ANSWER_SYSTEM = """You answer questions using the provided evidence.
CRITICAL RULES:
1. Pay close attention to staleness warnings on each document.
2. Prefer information from FRESH and CURRENT documents.
3. If a document is marked SUPERSEDED or VERY_STALE, warn the user and explicitly note
   that the information may be outdated.
4. If ALL retrieved documents are stale, state that clearly and recommend the user
   verify the information independently.
5. Cite documents by their ID: [D01], [D14], etc.
6. Include the document date when citing stale sources."""


def format_context_with_staleness(hits: list[dict]) -> str:
    """Format retrieved documents with staleness annotations for the LLM."""
    parts = []
    for h in hits:
        header = f"[{h['doc_id']}] {h['title']} (Date: {h['date']})"
        if h.get("staleness_warning"):
            header += f"\n  ⚠ STALENESS WARNING ({h['staleness']}): {h['staleness_warning']}"
        parts.append(f"{header}\n{h['text']}")
    return "\n\n---\n\n".join(parts)


def recency_qa(
    question: str,
    method: str = "recency_aware",
    verbose: bool = True,
) -> dict:
    """Full QA pipeline with recency awareness and staleness labelling."""
    t0 = time.time()

    # Retrieve
    if method == "similarity_only":
        hits = retrieve_similarity_only(question, top_k=4)
    elif method == "recency_aware":
        hits = retrieve_recency_aware(question, top_k=4)
    elif method == "current_only":
        hits = retrieve_filtered(question, current_only=True, top_k=4)
    else:
        hits = retrieve_similarity_only(question, top_k=4)

    # Label staleness
    hits = label_result_set(hits)

    # Build staleness summary
    risk_counts = Counter(h["staleness_risk"] for h in hits)
    freshest_age = min(h["age_days"] for h in hits) if hits else 0
    all_stale = all(h["staleness_risk"] in ("MEDIUM", "HIGH") for h in hits)

    # Format context
    context = format_context_with_staleness(hits)

    # Generate answer
    prompt = f"QUESTION: {question}\n\nDOCUMENTS:\n{context}\n\nAnswer the question using these documents."
    if all_stale:
        prompt += "\n\n⚠ ALL documents are STALE. Flag this prominently in your answer."

    answer = ask(prompt, system=ANSWER_SYSTEM, temperature=0.2)
    elapsed = time.time() - t0

    result = {
        "question": question,
        "method": method,
        "answer": answer,
        "hits": hits,
        "risk_counts": dict(risk_counts),
        "all_stale": all_stale,
        "freshest_age_days": freshest_age,
        "time_s": elapsed,
    }

    if verbose:
        print(f"  Method: {method}")
        for h in hits:
            icon = {"NONE": " ", "LOW": "~", "MEDIUM": "!", "HIGH": "X"}[h["staleness_risk"]]
            print(f"    [{icon}] {h['doc_id']} {h['date']} {h['staleness']:<12} score={h['final_score']:.3f}  {h['title'][:35]}")
        if all_stale:
            print(f"    ** ALL DOCUMENTS STALE — answer flagged **")

    return result


print("Recency-aware QA pipeline ready")

## 16. Demo — Recency-Aware Answers

In [ ]:
demo_questions = [
    "What is our current remote work policy?",
    "What authentication does our API use?",
    "What are the password requirements?",
]

print("RECENCY-AWARE QA DEMO")
print("=" * 100)

for q in demo_questions:
    print(f"\nQ: {q}")
    print("-" * 85)
    r = recency_qa(q, method="recency_aware")
    print(f"  Answer:")
    wrap_print("    " + r["answer"][:350])
    print(f"  Time: {r['time_s']:.1f}s")

---

# Part D — Comparative Evaluation

## 17. Evaluation Questions

Questions designed to test whether recency-awareness improves answer accuracy when documents have been superseded.

In [ ]:
EVAL_QUESTIONS = [
    # Questions where the latest version is critical
    {"question": "What is our current remote work policy?",
     "type": "superseded",
     "expected_doc": "D14",
     "trap_doc": "D01",
     "ground_truth_fragment": "flexible work is the default"},

    {"question": "What are the current password and authentication requirements?",
     "type": "superseded",
     "expected_doc": "D10",
     "trap_doc": "D02",
     "ground_truth_fragment": "16-character passphrases"},

    {"question": "How does our API authenticate requests?",
     "type": "superseded",
     "expected_doc": "D16",
     "trap_doc": "D04",
     "ground_truth_fragment": "mTLS"},

    {"question": "How is employee compensation structured?",
     "type": "superseded",
     "expected_doc": "D12",
     "trap_doc": "D03",
     "ground_truth_fragment": "location-based pay differentials eliminated"},

    {"question": "What are our data privacy compliance requirements?",
     "type": "superseded",
     "expected_doc": "D13",
     "trap_doc": "D05",
     "ground_truth_fragment": "GDPR, CCPA, and Brazil's LGPD"},

    {"question": "What is the incident response process for a major outage?",
     "type": "superseded",
     "expected_doc": "D17",
     "trap_doc": "D08",
     "ground_truth_fragment": "SEV0 added for security breaches"},

    # Questions where recency matters less (timeless content)
    {"question": "What are the code review requirements?",
     "type": "timeless",
     "expected_doc": "D07",
     "trap_doc": None,
     "ground_truth_fragment": "at least one approval"},

    # Questions about forward-looking content
    {"question": "What are the company's plans for 2025?",
     "type": "forward",
     "expected_doc": "D18",
     "trap_doc": None,
     "ground_truth_fragment": "AI-native features"},
]

print(f"Evaluation: {len(EVAL_QUESTIONS)} questions")
print(f"Types: {dict(Counter(q['type'] for q in EVAL_QUESTIONS))}")

## 18. Run Three Methods on All Questions

In [ ]:
methods = ["similarity_only", "recency_aware", "current_only"]
eval_results = []

print("Running evaluation...\n")

for i, item in enumerate(EVAL_QUESTIONS, 1):
    q = item["question"]
    row = {"question": q, "type": item["type"], "expected_doc": item["expected_doc"]}

    for method in methods:
        r = recency_qa(q, method=method, verbose=False)

        # Check if expected doc is in top results
        retrieved_ids = [h["doc_id"] for h in r["hits"]]
        expected_rank = None
        for rank, rid in enumerate(retrieved_ids, 1):
            if rid == item["expected_doc"]:
                expected_rank = rank
                break

        # Check if trap doc (older version) is returned instead
        trap_in_top = item["trap_doc"] in retrieved_ids[:2] if item["trap_doc"] else False

        # Check if ground truth fragment appears in answer
        has_ground_truth = item["ground_truth_fragment"].lower() in r["answer"].lower()

        row[f"{method}_rank"] = expected_rank
        row[f"{method}_trap"] = trap_in_top
        row[f"{method}_correct"] = has_ground_truth
        row[f"{method}_answer"] = r["answer"]

    eval_results.append(row)

    # Progress
    ranks = [row.get(f"{m}_rank", "-") for m in methods]
    corrs = ["+" if row.get(f"{m}_correct") else "-" for m in methods]
    print(f"  [{i}/{len(EVAL_QUESTIONS)}] {q[:50]:<52} ranks={ranks}  correct={corrs}")

## 19. Retrieval Quality Comparison

In [ ]:
print("RETRIEVAL QUALITY")
print("=" * 85)

for method in methods:
    # Expected doc found in top results
    found = sum(1 for r in eval_results if r.get(f"{method}_rank") is not None)
    # Expected doc is rank 1
    rank1 = sum(1 for r in eval_results if r.get(f"{method}_rank") == 1)
    # Trap doc (stale version) in top 2
    traps = sum(1 for r in eval_results if r.get(f"{method}_trap"))
    # Answer includes ground truth
    correct_answers = sum(1 for r in eval_results if r.get(f"{method}_correct"))

    print(f"\n  {method}:")
    print(f"    Expected doc found:       {found}/{len(eval_results)}")
    print(f"    Expected doc at rank 1:   {rank1}/{len(eval_results)}")
    print(f"    Stale trap doc in top 2:  {traps}/{len(eval_results)}")
    print(f"    Answer has ground truth:  {correct_answers}/{len(eval_results)} ({correct_answers/len(eval_results):.0%})")

# Focus on superseded questions (where recency matters most)
superseded = [r for r in eval_results if r["type"] == "superseded"]
print(f"\n  ON SUPERSEDED QUESTIONS ONLY ({len(superseded)} questions):")
for method in methods:
    rank1 = sum(1 for r in superseded if r.get(f"{method}_rank") == 1)
    correct = sum(1 for r in superseded if r.get(f"{method}_correct"))
    traps = sum(1 for r in superseded if r.get(f"{method}_trap"))
    print(f"    {method:<18} rank1={rank1}/{len(superseded)}  correct={correct}/{len(superseded)}  traps={traps}")

## 20. Side-by-Side Answer Comparison

In [ ]:
# Show 2 superseded questions where method differences are most visible
print("ANSWER COMPARISON (superseded topics)")
print("=" * 100)

for r in superseded[:2]:
    print(f"\nQ: {r['question']}")
    print(f"Expected: {r['expected_doc']} | Ground truth fragment: '{r.get('ground_truth_fragment', EVAL_QUESTIONS[0]['ground_truth_fragment'])[:40]}'")
    # Find in EVAL_QUESTIONS for ground_truth_fragment
    matching_q = next((eq for eq in EVAL_QUESTIONS if eq["question"] == r["question"]), None)
    if matching_q:
        print(f"Expected: {r['expected_doc']} | Ground truth: '{matching_q['ground_truth_fragment'][:50]}'")

    for method in methods:
        correct = "+" if r.get(f"{method}_correct") else "-"
        rank = r.get(f"{method}_rank", "-")
        trap = "TRAP!" if r.get(f"{method}_trap") else ""
        print(f"\n  [{correct}] {method} (expected doc rank={rank}) {trap}")
        wrap_print("      " + r[f"{method}_answer"][:200])
    print()

## 21. Recency Weight Sensitivity Analysis

How does changing the recency weight affect retrieval quality?

In [ ]:
# Test different recency weights on superseded questions
weights = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

print("RECENCY WEIGHT SENSITIVITY (superseded questions, retrieval only)")
print("=" * 75)

superseded_qs = [eq for eq in EVAL_QUESTIONS if eq["type"] == "superseded"]

print(f"{'Weight':<10} {'Expected@1':<12} {'Expected@Top':<14} {'Traps in Top2':<14}")
print("-" * 50)

for w in weights:
    at_1 = 0
    in_top = 0
    traps = 0

    for eq in superseded_qs:
        hits = retrieve_recency_aware(eq["question"], top_k=4, recency_weight=w)
        ids = [h["doc_id"] for h in hits]

        if ids and ids[0] == eq["expected_doc"]:
            at_1 += 1
        if eq["expected_doc"] in ids:
            in_top += 1
        if eq["trap_doc"] and eq["trap_doc"] in ids[:2]:
            traps += 1

    n = len(superseded_qs)
    print(f"{w:<10.1f} {at_1}/{n:<10} {in_top}/{n:<12} {traps}/{n}")

print(f"\nNote: weight=0.0 is pure similarity, weight=1.0 is pure recency.")
print(f"Sweet spot is typically 0.25-0.40 — enough to prefer recent docs without")
print(f"overriding genuinely relevant older content.")

## 22. LLM-as-Judge — Answer Quality

In [ ]:
JUDGE_SYSTEM = """You judge which answer to a question is more accurate and safe.
Penalize answers that present outdated information as current.
Reward answers that flag stale sources and prefer recent information."""

JUDGE_PROMPT = """QUESTION: {question}

ANSWER A ({method_a}):
{answer_a}

ANSWER B ({method_b}):
{answer_b}

Which answer is more accurate and less likely to mislead with outdated information?
Return ONLY JSON:
{{"winner": "A" or "B" or "tie",
  "reasoning": "brief explanation",
  "quality_a": 1-5,
  "quality_b": 1-5}}"""


print("LLM JUDGE: similarity_only vs recency_aware (superseded questions)")
print("=" * 85)

wins = {"similarity_only": 0, "recency_aware": 0, "tie": 0}

for r in superseded:
    q_short = textwrap.shorten(r["question"], 55, placeholder="...")

    j = parse_json(ask(
        JUDGE_PROMPT.format(
            question=r["question"],
            method_a="similarity_only",
            answer_a=r["similarity_only_answer"][:350],
            method_b="recency_aware",
            answer_b=r["recency_aware_answer"][:350],
        ),
        system=JUDGE_SYSTEM, temperature=0.0,
    )) or {"winner": "tie", "quality_a": 3, "quality_b": 3}

    w = (
        "similarity_only" if j.get("winner") == "A"
        else "recency_aware" if j.get("winner") == "B"
        else "tie"
    )
    wins[w] += 1

    qa = j.get("quality_a", "?")
    qb = j.get("quality_b", "?")
    print(f"  {q_short}")
    print(f"    winner: {w:<18} (sim={qa}/5, recency={qb}/5)")

print(f"\n  SUMMARY:")
print(f"    similarity_only wins: {wins['similarity_only']}")
print(f"    recency_aware wins:   {wins['recency_aware']}")
print(f"    ties:                 {wins['tie']}")

---

# Part E — Edge Cases & Analysis

## 23. Edge Case: All Retrieved Documents Are Stale

In [ ]:
# Ask a question where the answer topics mostly have old documents
edge_q = "What are the password rotation requirements?"
print(f"EDGE CASE: Question about a policy that changed significantly")
print(f"Q: {edge_q}")
print(f"\n  Old answer (2019): rotate every 90 days")
print(f"  Current answer (2023): rotation ELIMINATED per NIST guidelines")
print("-" * 80)

# Similarity-only might rank the old policy higher (more keywords match)
r_naive = recency_qa(edge_q, method="similarity_only")
print(f"\n  SIMILARITY-ONLY ANSWER:")
wrap_print("    " + r_naive["answer"][:250])

r_aware = recency_qa(edge_q, method="recency_aware")
print(f"\n  RECENCY-AWARE ANSWER:")
wrap_print("    " + r_aware["answer"][:250])

# This is a classic trap: the old doc literally says "password rotation every 90 days"
# which is a strong keyword match, but the current policy says rotation is eliminated.
eliminated_in_naive = "eliminated" in r_naive["answer"].lower() or "no longer" in r_naive["answer"].lower()
eliminated_in_aware = "eliminated" in r_aware["answer"].lower() or "no longer" in r_aware["answer"].lower()
print(f"\n  Mentions rotation eliminated: naive={eliminated_in_naive}, aware={eliminated_in_aware}")

## 24. Edge Case: Timeless Document

In [ ]:
# Code review standards (D07) from 2022 — still current, never superseded
edge_q2 = "What is the code review process?"
print(f"EDGE CASE: Timeless document (not superseded, but 2+ years old)")
print(f"Q: {edge_q2}")
print("-" * 80)

r_timeless = recency_qa(edge_q2, method="recency_aware")
print(f"\n  RECENCY-AWARE ANSWER:")
wrap_print("    " + r_timeless["answer"][:250])

# Check that the stale label correctly identifies it as aging but not superseded
code_review_hits = [h for h in r_timeless["hits"] if h["doc_id"] == "D07"]
if code_review_hits:
    h = code_review_hits[0]
    print(f"\n  D07 labels: staleness={h['staleness']}, risk={h['staleness_risk']}, is_current={h['is_current']}")
    print(f"  Correct behavior: flagged as aging but NOT superseded")

## 25. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Using only hard date filters | Loses relevant older content (e.g., timeless code review standards) | Soft decay scoring + staleness labels |
| Ignoring document dates entirely | Top-1 hit may be a superseded policy from 2019 | Include dates in metadata and scoring |
| Setting decay too aggressive | Recent but irrelevant docs beat old but relevant ones | Tune weight < 0.5; similarity should dominate |
| Not labelling staleness for the LLM | LLM presents old info as current because it doesn't know | Pass staleness warnings in the context |
| Treating all old docs as stale | Some docs (standards, principles) are valid for years | Use superseded_by flags alongside age |
| Not tracking document lineage | Can't tell if v1 was replaced by v2 | Model supersession chains: v1 → v2 → v3 |

## 26. Production Improvement Ideas

1. **Versioned document chains** — link superseded docs explicitly so the system always knows the current version
2. **Auto-expiry rules** — certain categories (policy, compliance) automatically flag after 12 months
3. **User-facing freshness badge** — show "Last updated: 6 months ago" on every answer source
4. **Dynamic decay per category** — security policies decay faster than engineering principles
5. **Staleness feedback loop** — track when users flag stale answers and retrain the decay model
6. **Multi-version synthesis** — when a document has v1/v2/v3, summarize what changed across versions

## 27. Exercises

### Exercise 1
Implement **category-specific decay rates**: security policies should decay 2x faster than engineering standards. Add a `DECAY_MULTIPLIER` dict mapping category → multiplier and modify `retrieve_recency_aware` to use it.

### Exercise 2
Build a **version diff summarizer**: when the retriever finds both v1 and v3 of a policy, generate a summary of what changed between versions. Use the LLM to produce a structured diff (added, removed, modified).

### Exercise 3
Add a **confidence meter** based on staleness: if the freshest document is > 6 months old, the answer should include an explicit confidence level (HIGH / MEDIUM / LOW) and a recommendation to verify with a subject matter expert.

### Mini Challenge
Implement a **time-aware query rewriter**: when the user asks "What's the current policy on X?", detect the temporal intent and automatically (1) set metadata filter for recent docs, (2) boost decay weight, and (3) add version-chain awareness to prefer the latest version in a supersession chain.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Corpus: {len(DOCUMENTS)} documents spanning {min(d['date'] for d in DOCUMENTS)} to {max(d['date'] for d in DOCUMENTS)}")
print(f"  Current: {sum(1 for d in DOCUMENTS if d['is_current'])} | Superseded: {sum(1 for d in DOCUMENTS if not d['is_current'])}")
print(f"  Categories: {dict(Counter(d['category'] for d in DOCUMENTS))}")
print()
print(f"Evaluation: {len(EVAL_QUESTIONS)} questions")
print(f"  recency_aware wins: {wins.get('recency_aware', 0)}/{len(superseded)} vs similarity_only")
print()
print("Components built:")
print("  1. Three decay functions     — exponential, linear, step")
print("  2. Similarity-only retriever — baseline with no recency awareness")
print("  3. Recency-aware retriever   — blended sim + decay scoring")
print("  4. Metadata-filtered retriever — ChromaDB where clauses")
print("  5. Staleness labeller        — FRESH / AGING / STALE / VERY_STALE / SUPERSEDED")
print("  6. Recency-aware QA          — LLM answers with staleness warnings injected")
print("  7. Weight sensitivity analysis — optimal recency_weight exploration")
print("  8. LLM-as-judge evaluation   — quality comparison across methods")

## 28. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Similarity-only retrieval is dangerous when documents evolve** — a superseded 2020 policy can outrank the current 2024 version |
| 2 | **Time-decay scoring blends relevance and freshness** — `score = (1-w) × sim + w × decay(age)` with weight ~0.3 works well |
| 3 | **Metadata filters are deterministic and efficient** — use them for hard constraints (current_only, min_year) |
| 4 | **Staleness labels must be passed to the LLM** — the model can't judge freshness without explicit age/status annotations |
| 5 | **Not all old docs are stale** — engineering principles from 2022 may still be fully valid; track `superseded_by` explicitly |
| 6 | **Exponential decay is preferred** — smooth, tunable via half-life, no hard cutoffs |
| 7 | **Keep recency weight below 0.5** — similarity should dominate; recency is a tiebreaker, not the primary signal |
| 8 | **When all sources are stale, say so** — worst outcome is presenting outdated info as current with high confidence |
| 9 | **Document version chains** are essential in production — knowing that D01 → D06 → D14 lets the system always find the latest |
| 10 | **Recency awareness is a safety feature** — in compliance, finance, and medicine, outdated info can cause real harm |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*